# Road Network Quality Index (RNQI)

This notebook evaluates and ranks city road networks using connectivity, efficiency, and hierarchy-based indicators.


## 1. Setup and imports
This cell loads the Python libraries required for geospatial processing, graph analysis, and file handling.


In [14]:
import os
import time
import random
import math
import json
import requests

import numpy as np
import pandas as pd
import networkx as nx

## 2. Reference city locations
This cell defines the benchmark cities and their latitude/longitude coordinates for comparison.


In [52]:
reference_places = {
    # Chandigarh
    "Chandigarh_Burail": {
        "city": "Chandigarh",
        "typology": "Organic Core",
        "lat": 30.7050, "lon": 76.7612,  # Centered on Burail Village center (Sector 45)
        "label": "Burail / Sector 45",
    },
    "Chandigarh_Sector17": {
        "city": "Chandigarh",
        "typology": "Planned CBD",
        "lat": 30.7410, "lon": 76.7792,  # Sector 17 Shopping Plaza Core
        "label": "Sector 17 Plaza",
    },
    "Chandigarh_ITPark": {
        "city": "Chandigarh",
        "typology": "Peripheral",
        "lat": 30.7265, "lon": 76.8441,  # Centered on DLF Infocity / Main Spine Road
        "label": "IT Park, Kishangarh / Manimajra",
    },

    # Pune
    "Pune_Peth": {
        "city": "Pune",
        "typology": "Organic Core",
        "lat": 18.5208, "lon": 73.8515,  # Centered near the historical Shaniwar Wada axis
        "label": "Shaniwar Peth",
    },
    "Pune_Cantonment": {
        "city": "Pune",
        "typology": "Planned CBD",
        "lat": 18.5145, "lon": 73.8778,  # Anchored on MG Road (Planned Military Grid)
        "label": "Camp / Koregaon Park",
    },
    "Pune_Hinjawadi": {
        "city": "Pune",
        "typology": "Peripheral",
        "lat": 18.5947, "lon": 73.7095,  # Shivaji Chowk, Phase 1 Arterial Spine
        "label": "Hinjawadi Phase 1",
    },

    # Kolkata
    "Kolkata_Burrabazar": {
        "city": "Kolkata",
        "typology": "Organic Core",
        "lat": 22.5820, "lon": 88.3520,  # Centered along Mahatma Gandhi Road organic zone
        "label": "Burrabazar",
    },
    "Kolkata_NewTown": {
        "city": "Kolkata",
        "typology": "Planned CBD",
        "lat": 22.5894, "lon": 88.4748,  # Major grid intersection of Action Area 1
        "label": "New Town Action Area 1",
    },
    "Kolkata_SectorV": {
        "city": "Kolkata",
        "typology": "Peripheral",
        "lat": 22.5755, "lon": 88.4323,  # Tech-park axis at College More
        "label": "Salt Lake Sector V",
    },

    # Jaipur
    "Jaipur_PinkCity": {
        "city": "Jaipur",
        "typology": "Organic Core",
        "lat": 26.9200, "lon": 75.8235,  # Symmetrical axis of Badi Choupad
        "label": "Pink City, Badi Choupad",
    },
    "Jaipur_CScheme": {
        "city": "Jaipur",
        "typology": "Planned CBD",
        "lat": 26.9040, "lon": 75.7986,  # Centered on the planned grid of Ashok Nagar
        "label": "C-Scheme, Ashok Nagar",
    },
    "Jaipur_Sitapura": {
        "city": "Jaipur",
        "typology": "Peripheral",
        "lat": 26.7820, "lon": 75.8250,  # Central block of the heavy industrial loop
        "label": "Sitapura Industrial Area",
    },

    # Ranipet
    "Ranipet_OldTown": {
        "city": "Ranipet",
        "typology": "Organic Core",
        "lat": 12.9321, "lon": 79.3335,  # Centered on the core historic settlement near Palar River
        "label": "Ranipet Old Town",
    },
    "Ranipet_BHEL": {
        "city": "Ranipet",
        "typology": "Planned CBD",
        "lat": 12.9705, "lon": 79.2848,  # Center of the highly structured planned layout
        "label": "BHEL Township",
    },
    "Ranipet_SIPCOT": {
        "city": "Ranipet",
        "typology": "Peripheral",
        "lat": 12.9623, "lon": 79.3064,  # Industrial super-blocks axis
        "label": "SIPCOT Phase 1",
    },
}


## 3. Study area settings
This cell sets the analysis radius, buffer distance, cache folder, and output file for all city evaluations.


In [53]:
STUDY_RADIUS_METERS = 5000
BUFFER_METERS = 150
API_DELAY_SECONDS = 3

CACHE_DIR = (
    "/Users/ankita/Documents/masters/"
    "RoadNetworkOutputs/New_code/"
    "rnqi_radius_1500m_buffer_150m_consolidated"
)
PBF_PATH = "/Users/ankita/Documents/masters/RoadNetworkOutputs/New_code/india-260913.osm.pbf"

os.makedirs(CACHE_DIR, exist_ok=True)

OUTPUT_PATH = (
    "/Users/ankita/Documents/masters/"
    "RoadNetworkOutputs/New_code/"
    "rnqi_typology_metrics.csv"
)

## 4. Overpass graph download
This cell defines a failover mechanism to download road network data from multiple OpenStreetMap endpoints.


In [54]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

HEADERS = {
    "User-Agent": "UrbanConnectivityResearch/3.0"
}

RADIUS = 1500

def download_graph_with_failover(
    boundary_polygon,
    retries_per_endpoint=1,
):
    """Download a drive graph with bounded retries and failover."""

    original_endpoint = ox.settings.overpass_url
    original_timeout = ox.settings.requests_timeout
    last_error = None

    ox.settings.requests_timeout = 120
    ox.settings.overpass_settings = (
        "[out:json][timeout:120][maxsize:1073741824]"
    )

    try:
        for endpoint in OVERPASS_ENDPOINTS:
            ox.settings.overpass_url = endpoint

            for attempt in range(1, retries_per_endpoint + 1):
                try:
                    print(
                        f"Querying {endpoint} "
                        f"(attempt {attempt})"
                    )

                    graph = ox.graph_from_polygon(
                        boundary_polygon,
                        network_type="drive",
                        simplify=True,
                    )

                    print("Download successful.")
                    return graph

                except Exception as error:
                    last_error = error

                    print(
                        f"Failed: {type(error).__name__}: {error}"
                    )

                    if attempt < retries_per_endpoint:
                        time.sleep(10)

            print(
                f"Moving to the next endpoint after "
                f"{endpoint} failed."
            )
            time.sleep(10)

    finally:
        ox.settings.overpass_url = original_endpoint
        ox.settings.requests_timeout = original_timeout

    raise RuntimeError(
        "All Overpass endpoints failed. "
        f"Last error: {last_error}"
    )

## 5. Study-area geometry
This cell creates the circular analysis polygon and converts it between geographic and projected coordinate systems.


In [55]:
def create_study_area(
    lat,
    lon,
    radius_meters,
):
    """
    Create circular study area.

    Returns:
        study_polygon_wgs84
        study_polygon_projected
        projected_crs
    """

    point_gdf = gpd.GeoDataFrame(
        geometry=[Point(lon, lat)],
        crs="EPSG:4326",
    )

    point_projected = ox.projection.project_gdf(
        point_gdf
    )

    projected_crs = point_projected.crs

    study_polygon_projected = (
        point_projected.geometry.iloc[0]
        .buffer(radius_meters)
    )

    study_polygon_wgs84 = (
        gpd.GeoSeries(
            [study_polygon_projected],
            crs=projected_crs,
        )
        .to_crs("EPSG:4326")
        .iloc[0]
    )

    return (
        study_polygon_wgs84,
        study_polygon_projected,
        projected_crs,
    )

## 6. Download or load city road graph
This cell fetches the road network for each city, caches it locally, and clips it to the actual study area.


In [56]:
def get_city_graph(zone, place_info):
    lat = place_info["lat"]
    lon = place_info["lon"]

    query = f"""
[out:json][timeout:90];
(
  way(around:{RADIUS},{lat},{lon})
    ["highway"]
    ["highway"!~"footway|pedestrian|path|service|cycleway|steps"];
);
out body;
>;
out skel qt;
"""

    last_error = None

    for endpoint in OVERPASS_ENDPOINTS:
        for attempt in range(3):
            try:
                print(
                    f"Requesting {zone} from {endpoint} "
                    f"(attempt {attempt + 1})"
                )

                response = requests.post(
                    endpoint,
                    data={"data": query},
                    headers=HEADERS,
                    timeout=120,
                )
                response.raise_for_status()

                data = response.json()
                graph = nx.MultiDiGraph()

                for element in data.get("elements", []):
                    if element["type"] == "node":
                        graph.add_node(
                            element["id"],
                            x=element["lon"],
                            y=element["lat"],
                        )

                for element in data.get("elements", []):
                    if element["type"] != "way":
                        continue

                    node_ids = element.get("nodes", [])
                    highway = element.get("tags", {}).get("highway")

                    for start, end in zip(node_ids, node_ids[1:]):
                        if start not in graph or end not in graph:
                            continue

                        graph.add_edge(
                            start,
                            end,
                            length=haversine_distance(
                                graph.nodes[start]["y"],
                                graph.nodes[start]["x"],
                                graph.nodes[end]["y"],
                                graph.nodes[end]["x"],
                            ),
                            highway=highway,
                        )

                        graph.add_edge(
                            end,
                            start,
                            length=haversine_distance(
                                graph.nodes[end]["y"],
                                graph.nodes[end]["x"],
                                graph.nodes[start]["y"],
                                graph.nodes[start]["x"],
                            ),
                            highway=highway,
                        )

                if graph.number_of_nodes() == 0:
                    raise RuntimeError("Overpass returned no road nodes.")

                print(
                    f"Loaded {graph.number_of_nodes()} nodes and "
                    f"{graph.number_of_edges()} edges."
                )

                return graph, (math.pi * RADIUS**2) / 1_000_000, None

            except Exception as error:
                last_error = error
                print(f"Request failed: {error}")
                time.sleep(5)

        time.sleep(10)

    raise RuntimeError(
        f"All Overpass endpoints failed for {zone}: {last_error}"
    )

def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius_meters = 6_371_000

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    value = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(delta_lon / 2) ** 2
    )

    return 2 * earth_radius_meters * math.asin(math.sqrt(value))

## 7. Connectivity metrics
This cell computes node density, link density, degree, and sampled closeness to capture network connectivity.


In [57]:
def calculate_connectivity_metrics(
    graph,
    area_sq_km,
    seed=42,
    closeness_sample_size=20,
):
    """
    Calculate Connectivity pillar metrics.
    """

    node_count = graph.number_of_nodes()

    if node_count == 0:
        raise RuntimeError("Graph has no nodes.")

    edge_records = [
        {
            "u": source,
            "v": target,
            **attributes,
        }
        for source, target, attributes in graph.edges(data=True)
    ]

    edges = pd.DataFrame(edge_records)

    if edges.empty:
        raise RuntimeError("Graph has no edges.")

    edge_count = graph.number_of_edges()

    road_length_km = (
        edges["length"].sum() / 1000
    )

    average_degree = (
        sum(dict(graph.degree()).values())
        / node_count
    )

    # Sampled closeness
    sample_size = min(
        closeness_sample_size,
        node_count,
    )

    rng = random.Random(seed)

    sample_nodes = rng.sample(
        list(graph.nodes()),
        sample_size,
    )

    closeness_values = []

    for node in sample_nodes:

        lengths = (
            nx.single_source_dijkstra_path_length(
                graph,
                node,
                weight="length",
            )
        )

        reachable_lengths = [
            distance
            for target, distance in lengths.items()
            if target != node
        ]

        if reachable_lengths:

            closeness_values.append(
                len(reachable_lengths)
                / sum(reachable_lengths)
            )
    undirected_graph = nx.Graph(graph)

    dead_end_nodes = sum(
            degree == 1
            for _, degree in undirected_graph.degree()
        )

    dead_end_percentage = (
            100 * dead_end_nodes / node_count
        )

    edge_table = edges.reset_index()

    edge_table["node_a"] = edge_table[["u", "v"]].min(axis=1)
    edge_table["node_b"] = edge_table[["u", "v"]].max(axis=1)

    unique_segments = (
            edge_table
            .groupby(["node_a", "node_b"], as_index=False)["length"]
            .max()
        )

    average_segment_length = (
            unique_segments["length"].mean()
        )
        
    return {
        "Nodes": node_count,
        "Edges": edge_count,
        "Node Density": (
            node_count / area_sq_km
        ),
        "Link Density": (
            road_length_km / area_sq_km
        ),
        "Average Degree": average_degree,
        "Mean Sampled Closeness": (
            np.mean(closeness_values)
            if closeness_values
            else np.nan
        ),
        "Total Road Length (km)": road_length_km,
        "Dead-End Percentage": dead_end_percentage,
        "Average Segment Length (m)": average_segment_length,
        "Unique Street Segments": len(unique_segments),
    }

## 8. Efficiency metrics
This cell estimates network efficiency using shortest-path lengths, inverse-efficiency values, and circuity.


In [58]:
def calculate_efficiency_metrics(
    graph,
    sample_size=40,
    seed=42,
):
    """
    Calculate Efficiency pillar metrics.

    Uses a sampled approximation for large graphs.
    """

    nodes = list(graph.nodes())

    if len(nodes) < 2:
        raise RuntimeError(
            "At least 2 nodes required."
        )

    rng = random.Random(seed)

    sample_nodes = rng.sample(
        nodes,
        min(sample_size, len(nodes)),
    )

    shortest_path_lengths = []
    efficiency_values = []
    circuity_values = []

    for source in sample_nodes:

        lengths = (
            nx.single_source_dijkstra_path_length(
                graph,
                source,
                weight="length",
            )
        )

        for target, distance in lengths.items():

            if target == source:
                continue

            if distance > 0:

                shortest_path_lengths.append(
                    distance
                )

                efficiency_values.append(
                    1 / distance
                )

                # Euclidean distance
                x1 = graph.nodes[source]["x"]
                y1 = graph.nodes[source]["y"]

                x2 = graph.nodes[target]["x"]
                y2 = graph.nodes[target]["y"]

                euclidean_distance = haversine_distance(
                            y1,
                            x1,
                            y2,
                            x2,
                        )

                if euclidean_distance > 0:

                    circuity_values.append(
                        distance
                        / euclidean_distance
                    )

    return {
        "Global Efficiency": (
            np.mean(efficiency_values)
            if efficiency_values
            else np.nan
        ),

        "ASP Length (m)": (
            np.mean(shortest_path_lengths)
            if shortest_path_lengths
            else np.nan
        ),

        "Circuity": (
            np.mean(circuity_values)
            if circuity_values
            else np.nan
        ),

        "Efficiency Sample Size": len(
            shortest_path_lengths
        ),
    }

## 9. Hierarchy metrics
This cell measures betweenness centrality and road class distribution to assess structural hierarchy.


In [59]:
def calculate_hierarchy_metrics(
    graph,
    sample_size=40,
):
    """
    Calculate hierarchy metrics from sampled
    betweenness centrality and road classes.
    """

    node_count = graph.number_of_nodes()

    if node_count == 0:
        raise RuntimeError("Graph has no nodes.")

    k = min(sample_size, node_count)

    betweenness = nx.betweenness_centrality(
        graph,
        k=k,
        weight="length",
        normalized=True,
        seed=42,
    )

    betweenness_values = np.asarray(
        list(betweenness.values()),
        dtype=float,
    )

    betweenness_variance = np.var(
        betweenness_values,
        ddof=0,
    )

    road_class_counts = {}

    for _, _, attributes in graph.edges(data=True):
        highway_value = attributes.get("highway")

        if highway_value is None:
            continue

        if isinstance(highway_value, list):
            highway_value = highway_value[0]

        road_class_counts[highway_value] = (
            road_class_counts.get(highway_value, 0) + 1
        )

    return {
        "Betweenness Variance": betweenness_variance,
        "Road Class Distribution": str(
            road_class_counts
        ),
    }

## 10. Hierarchy clarity placeholder
This cell defines a preliminary indicator for road-class diversity and leaves room for a formal hierarchy-clarity formula.


In [60]:
def calculate_hierarchy_clarity(edges):
    """
    Placeholder for hierarchy clarity.

    Needs a formally defined formula before
    inclusion in final RNQI.
    """

    if "highway" not in edges.columns:
        return np.nan

    valid_classes = []

    for value in edges["highway"].dropna():

        if isinstance(value, list):
            value = value[0]

        valid_classes.append(value)

    if not valid_classes:
        return np.nan

    unique_classes = len(
        set(valid_classes)
    )

    return unique_classes

## 11. Compute all city metrics
This cell runs the full metric pipeline for each city and compiles the raw RNQI values into a result dictionary.


In [61]:
def calculate_city_metrics(
    zone,
    place_info,
):
    """
    Calculate raw RNQI metrics for one city.
    """

    print(f"\nCalculating {zone}...")

    graph, area_sq_km, study_polygon = get_city_graph(
        zone,
        place_info,
    )

    # Keep largest weakly connected component
    largest_component = max(
        nx.weakly_connected_components(graph),
        key=len,
    )

    analysis_graph = graph.subgraph(
        largest_component
    ).copy()

    connectivity = calculate_connectivity_metrics(
        analysis_graph,
        area_sq_km,
    )

    efficiency = calculate_efficiency_metrics(
        analysis_graph,
    )

    hierarchy = calculate_hierarchy_metrics(
        analysis_graph,
    )

    return {
        "Zone": zone,
        "City": place_info["city"],
        "Typology": place_info["typology"],
        "Label": place_info["label"],
        "Latitude": place_info["lat"],
        "Longitude": place_info["lon"],
        "Study Area (km²)": area_sq_km,
        **connectivity,
        **efficiency,
        **hierarchy,
    }


## 12. Run city-by-city analysis
This cell iterates through all reference cities, executes the metric calculations, and saves each result to the output CSV.


In [62]:
city_results = []
city_errors = []

for index, (zone, place_info) in enumerate(
    reference_places.items(),
    start=1,
):
    print(
        f"\n[{index}/{len(reference_places)}] "
        f"Processing {zone}..."
    )

    try:
        result = calculate_city_metrics(
            zone,
            place_info,
        )

        city_results.append(result)
        print(f"Completed {zone}")

    except Exception as error:
        city_errors.append(
            {
                "Zone": zone,
                "City": place_info["city"],
                "Typology": place_info["typology"],
                "Error": str(error),
            }
        )

        print(
            f"Failed {zone}: "
            f"{type(error).__name__}: {error}"
        )

    time.sleep(API_DELAY_SECONDS)

results_df = pd.DataFrame(city_results)

results_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

if city_errors:
    errors_path = os.path.join(
        os.path.dirname(OUTPUT_PATH),
        "rnqi_typology_errors.csv",
    )

    pd.DataFrame(city_errors).to_csv(
        errors_path,
        index=False,
    )

print(
    f"\nCompleted: {len(city_results)} "
    f"of {len(reference_places)} locations."
)

if city_errors:
    print(f"Failed locations: {len(city_errors)}")
    display(pd.DataFrame(city_errors))
else:
    print("All locations completed successfully.")


[1/15] Processing Chandigarh_Burail...

Calculating Chandigarh_Burail...
Requesting Chandigarh_Burail from https://overpass-api.de/api/interpreter (attempt 1)
Loaded 3828 nodes and 8884 edges.
Completed Chandigarh_Burail

[2/15] Processing Chandigarh_Sector17...

Calculating Chandigarh_Sector17...
Requesting Chandigarh_Sector17 from https://overpass-api.de/api/interpreter (attempt 1)
Request failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
Requesting Chandigarh_Sector17 from https://overpass-api.de/api/interpreter (attempt 2)
Request failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
Requesting Chandigarh_Sector17 from https://overpass-api.de/api/interpreter (attempt 3)
Loaded 2509 nodes and 5854 edges.
Completed Chandigarh_Sector17

[3/15] Processing Chandigarh_ITPark...

Calculating Chandigarh_ITPark...
Requesting Chandigarh_ITPark from https://overpass-api.de/api/interpreter (attempt 1)
Loaded 2450 no

## 13. Normalization helpers
This cell defines min-max scaling functions used to convert raw metric values into comparable scores.


In [63]:
def robust_minmax_normalize(
    series,
    log_transform=True,
    lower_quantile=0.05,
    upper_quantile=0.95,
    score_floor=5.0,
    score_ceiling=95.0,
):
    """
    Scale a metric to a bounded 5-95 range after reducing
    skew and clipping extreme values at robust quantile bounds.
    """

    values = pd.to_numeric(series, errors="coerce")

    if log_transform:
        values = np.log1p(values.clip(lower=0))

    lower_bound = values.quantile(lower_quantile)
    upper_bound = values.quantile(upper_quantile)

    if pd.isna(lower_bound) or pd.isna(upper_bound):
        return pd.Series(np.nan, index=series.index)

    if upper_bound == lower_bound:
        return pd.Series(
            (score_floor + score_ceiling) / 2,
            index=series.index,
        )

    clipped_values = values.clip(
        lower=lower_bound,
        upper=upper_bound,
    )

    normalized_values = (
        (clipped_values - lower_bound)
        / (upper_bound - lower_bound)
    )

    return (
        score_floor
        + normalized_values
        * (score_ceiling - score_floor)
    )


## 14. Inverse normalization helper
This cell creates the inverse min-max function so lower-is-better indicators like circuity and ASP are scored positively.


In [64]:
def robust_minmax_normalize_inverse(
    series,
    log_transform=True,
    lower_quantile=0.05,
    upper_quantile=0.95,
    score_floor=5.0,
    score_ceiling=95.0,
):
    """
    Scale a lower-is-better metric to a bounded 5-95 range
    after reducing skew and clipping extreme values.
    """

    values = pd.to_numeric(series, errors="coerce")

    if log_transform:
        values = np.log1p(values.clip(lower=0))

    lower_bound = values.quantile(lower_quantile)
    upper_bound = values.quantile(upper_quantile)

    if pd.isna(lower_bound) or pd.isna(upper_bound):
        return pd.Series(np.nan, index=series.index)

    if upper_bound == lower_bound:
        return pd.Series(
            (score_floor + score_ceiling) / 2,
            index=series.index,
        )

    clipped_values = values.clip(
        lower=lower_bound,
        upper=upper_bound,
    )

    normalized_values = (
        (upper_bound - clipped_values)
        / (upper_bound - lower_bound)
    )

    return (
        score_floor
        + normalized_values
        * (score_ceiling - score_floor)
    )


## 15. Convert metrics to scores
This cell normalizes each raw metric into a 0–100 score so all pillars can be compared fairly.


In [65]:
results_df = pd.DataFrame(city_results)

# Connectivity: log-transform positive, right-skewed metrics.
results_df["Node Density Score"] = (
    robust_minmax_normalize(
        results_df["Node Density"]
    )
)

results_df["Link Density Score"] = (
    robust_minmax_normalize(
        results_df["Link Density"]
    )
)

results_df["Average Degree Score"] = (
    robust_minmax_normalize(
        results_df["Average Degree"]
    )
)

results_df["Closeness Score"] = (
    robust_minmax_normalize(
        results_df["Mean Sampled Closeness"]
    )
)

# Efficiency: use the inverse form for lower-is-better metrics.
results_df["Global Efficiency Score"] = (
    robust_minmax_normalize(
        results_df["Global Efficiency"]
    )
)

results_df["ASP Score"] = (
    robust_minmax_normalize_inverse(
        results_df["ASP Length (m)"]
    )
)

results_df["Circuity Score"] = (
    robust_minmax_normalize_inverse(
        results_df["Circuity"]
    )
)

# Hierarchy: score concentration of structural importance.
results_df["Betweenness Variance Score"] = (
    robust_minmax_normalize(
        results_df["Betweenness Variance"]
    )
)


## 16. Pillar score grouping
This cell defines the metrics that belong to connectivity, efficiency, and hierarchy before aggregating each pillar score.


In [66]:
connectivity_columns = [
    "Node Density Score",
    "Link Density Score",
    "Average Degree Score",
    "Closeness Score",
]

efficiency_columns = [
    "Global Efficiency Score",
    "ASP Score",
    "Circuity Score",
]

hierarchy_columns = [
    "Betweenness Variance Score",
]


## 17. Aggregate pillar scores
This cell combines the normalized sub-scores for connectivity, efficiency, and hierarchy into separate pillar totals.


In [67]:
results_df["Connectivity Score"] = (
    results_df[connectivity_columns].mean(axis=1)
)

results_df["Efficiency Score"] = (
    results_df[efficiency_columns].mean(axis=1)
)

results_df["Hierarchy Score"] = (
    results_df[hierarchy_columns].mean(axis=1)
)

## 18. Final RNQI calculation
This cell computes the final composite Road Network Quality Index by averaging the three pillar scores.


In [68]:
# Weighted RNQI: connectivity and efficiency are given greater importance,
# while hierarchy remains a supporting structural measure.
connectivity_weight = 1
efficiency_weight = 1
hierarchy_weight = 1

results_df["RNQI Score"] = (
    results_df["Connectivity Score"] * connectivity_weight
    + results_df["Efficiency Score"] * efficiency_weight
    + results_df["Hierarchy Score"] * hierarchy_weight
) / (
    connectivity_weight
    + efficiency_weight
    + hierarchy_weight
)

results_df = results_df.sort_values(
    "RNQI Score",
    ascending=False,
).reset_index(drop=True)

## 19. Save and display ranking
This cell writes the final scores to CSV and presents the city ranking summary for the final RNQI comparison.


In [69]:
final_scores_path = os.path.join(
    os.path.dirname(OUTPUT_PATH),
    "rnqi_final_scores.csv",
)

results_df.to_csv(
    final_scores_path,
    index=False,
)

results_df[
    [
        "Zone",
        "City",
        "Typology",
        "Label",
        "Latitude",
        "Longitude",
        "Connectivity Score",
        "Efficiency Score",
        "Hierarchy Score",
        "RNQI Score",
    ]
].round(2)

,Zone,City,Typology,Label,Latitude,Longitude,Connectivity Score,Efficiency Score,Hierarchy Score,RNQI Score
0,Ranipet_BHEL,Ranipet,Planned CBD,BHEL Township,12.97,79.28,27.83,78.87,95.00,67.23
1,Pune_Hinjawadi,Pune,Peripheral,Hinjawadi Phase 1,18.59,73.71,19.85,77.20,93.28,63.45
2,Pune_Cantonment,Pune,Planned CBD,Camp / Koregaon Park,18.51,73.88,73.25,85.14,6.48,54.95
3,Pune_Peth,Pune,Organic Core,Shaniwar Peth,18.52,73.85,81.03,75.42,5.00,53.82
4,Jaipur_CScheme,Jaipur,Planned CBD,"C-Scheme, Ashok Nagar",26.90,75.80,71.00,55.69,18.45,48.38
5,Jaipur_PinkCity,Jaipur,Organic Core,"Pink City, Badi Choupad",26.92,75.82,72.62,55.93,5.92,44.82
6,Chandigarh_ITPark,Chandigarh,Peripheral,"IT Park, Kishangarh / Manimajra",30.73,76.84,57.72,44.15,31.59,44.49
7,Kolkata_Burrabazar,Kolkata,Organic Core,Burrabazar,22.58,88.35,52.66,59.01,17.73,43.13
8,Jaipur_Sitapura,Jaipur,Peripheral,Sitapura Industrial Area,26.78,75.82,60.66,29.89,36.53,42.36
9,Ranipet_OldTown,Ranipet,Organic Core,Ranipet Old Town,12.93,79.33,53.57,48.15,23.04,41.58


In [70]:
city_ranking = (
    results_df
    .groupby("City", as_index=False)
    .agg(
        Zones=("Zone", "count"),
        Connectivity_Score=("Connectivity Score", "mean"),
        Efficiency_Score=("Efficiency Score", "mean"),
        Hierarchy_Score=("Hierarchy Score", "mean"),
        RNQI_Score=("RNQI Score", "mean"),
    )
)

city_ranking["City Rank"] = (
    city_ranking["RNQI_Score"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

city_ranking = city_ranking.sort_values(
    "RNQI_Score",
    ascending=False,
).reset_index(drop=True)

city_ranking[
    [
        "City Rank",
        "City",
        "Zones",
        "Connectivity_Score",
        "Efficiency_Score",
        "Hierarchy_Score",
        "RNQI_Score",
    ]
].round(2)

,City Rank,City,Zones,Connectivity_Score,Efficiency_Score,Hierarchy_Score,RNQI_Score
0,1,Pune,3,58.04,79.25,34.92,57.41
1,2,Ranipet,3,37.96,44.79,54.59,45.78
2,3,Jaipur,3,68.09,47.17,20.30,45.19
3,4,Chandigarh,3,65.54,28.49,18.06,37.36
4,5,Kolkata,3,48.15,35.40,27.27,36.94


## 20. Interpretation and conclusion

### Zone-wise conclusion

The zone-wise RNQI ranking compares the 15 reference zones individually. A higher RNQI score indicates a stronger combined performance across connectivity, efficiency, and hierarchy. The ranking should be interpreted together with the `Zone`, `Typology`, `Label`, `Latitude`, and `Longitude` columns so that each score remains linked to its actual study location.

The zone-level results show that road-network quality varies substantially within the same city. This means that a city should not be treated as having one uniform road-network condition. Its Organic Core, Planned CBD, and Peripheral zone can perform differently because of differences in street density, road connectivity, route efficiency, and structural hierarchy.

The highest-ranked zones represent the strongest local network conditions in this analysis, while the lowest-ranked zones identify locations that may need targeted planning attention. These lower-ranked zones should be examined by pillar score: a low Connectivity Score points to weaker network density or access, a low Efficiency Score points to longer or more circuitous routes, and a low Hierarchy Score points to weaker concentration or organization of important routes.

### City-wise conclusion

The city-wise ranking is calculated by averaging the RNQI scores of the three reference zones for each city. This provides a balanced comparison across Chandigarh, Pune, Kolkata, Jaipur, and Ranipet because every city contributes one Organic Core, one Planned CBD, and one Peripheral zone.

A city with a high average RNQI score has relatively strong road-network performance across its different urban typologies. A city with a lower average score may have weaker overall performance or greater variation between its zones. Therefore, the city ranking represents the average condition of the sampled zones, not the quality of every road in the entire city.

The difference between a city's highest and lowest zone score is also important. A small difference suggests more consistent road-network performance across typologies, whereas a large difference indicates uneven development and a need for location-specific interventions. The final interpretation should therefore report both the city rank and the zone-level variation behind that rank.

Overall, the zone-wise ranking identifies where intervention is most needed, while the city-wise ranking summarizes how the five cities compare on average. Together, they provide a more informative assessment than either ranking alone.
